In [2]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.utils import resample

/kaggle/input/datasets/amanalisiddiqui/fraud-detection-dataset/AIML Dataset.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
df_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
# preserve all transactions
df = df_trans.merge(df_id, on='TransactionID', how='left')

**DATASET**

In [4]:
print(df.head())

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceType  \
0        NaN             NaN    NaN   Na

In [5]:
print(df.columns.tolist)
print(f"Size of transaction dataset: {df_trans.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Size of identity dataset: {df_id.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

<bound method IndexOpsMixin.tolist of Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)>
Size of transaction dataset: 2062.07 MB
Size of identity dataset: 143.14 MB


**EDA**

In [6]:
print(df['isFraud'].value_counts(normalize=True))

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [7]:
dtype_df = pd.DataFrame(df.dtypes.reset_index())
dtype_df.columns = ['Column', 'Data Type']
print(dtype_df)

             Column Data Type
0     TransactionID     int64
1           isFraud     int64
2     TransactionDT     int64
3    TransactionAmt   float64
4         ProductCD    object
..              ...       ...
429           id_36    object
430           id_37    object
431           id_38    object
432      DeviceType    object
433      DeviceInfo    object

[434 rows x 2 columns]


In [8]:
print(df.dtypes.value_counts())

float64    399
object      31
int64        4
Name: count, dtype: int64


In [9]:
def eda(df, target='isFraud'):
    results = {}

    missing_pct = df.isna().mean() * 100
    high_missing = missing_pct[missing_pct > 50].index.tolist()
    results['high_missing_cols'] = high_missing
    print(f"Columns with >50% missing: {high_missing}")

    return results
    

**TIME-BASED SPLIT**

In [10]:
df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

"""
For big datasets, n-splits = 10 is a good trafeoff between the size 
of each training set and the number of prediction sets
"""
tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    print(f"Fold {fold}: Train rows = {len(X_train)}, Test rows = {len(X_test)}, Fraud in test = {y_test.sum()}")

df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
print(f"\nTotal test rows across all folds: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))])}")
print(f"Average test rows per fold: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))]) / tscv.n_splits:.0f}")

Fold 1: Train rows = 53690, Test rows = 53685, Fraud in test = 1234
Fold 2: Train rows = 107375, Test rows = 53685, Fraud in test = 1727
Fold 3: Train rows = 161060, Test rows = 53685, Fraud in test = 2145
Fold 4: Train rows = 214745, Test rows = 53685, Fraud in test = 2376
Fold 5: Train rows = 268430, Test rows = 53685, Fraud in test = 2020
Fold 6: Train rows = 322115, Test rows = 53685, Fraud in test = 1856
Fold 7: Train rows = 375800, Test rows = 53685, Fraud in test = 2271
Fold 8: Train rows = 429485, Test rows = 53685, Fraud in test = 1940
Fold 9: Train rows = 483170, Test rows = 53685, Fraud in test = 1636
Fold 10: Train rows = 536855, Test rows = 53685, Fraud in test = 2006

Total test rows across all folds: 536850
Average test rows per fold: 53685


**SCALING**

In [11]:
"""
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
"""

'\nscaler = RobustScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n'